In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from datasets import load_dataset
from torch.utils.data import DataLoader
from collections import Counter

#Using gpu or cpu and print it
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Dataset Loading for LSTM Model

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")
train_data = dataset["train"]
test_data = dataset["test"]

Tokenization Function

In [ ]:
def tokenize(text):
    text = text.lower()      # یکسان‌سازی
    return text.split()      # جدا کردن کلمات

Vocabulary Construction and Token Frequency Counting

In [ ]:
counter = Counter()

# شمارش کلمات پرتکرار
for item in train_data:
    counter.update(tokenize(item["text"]))

# ساخت واژگان 20 هزار تایی
vocab = {word: i+2 for i, (word, _) in enumerate(counter.most_common(20000))}

# دو توکن خاص
vocab["<pad>"] = 0
vocab["<unk>"] = 1

Encoding Function for Token-to-ID Conversion

In [ ]:
def encode(text):
    tokens = tokenize(text)
    ids = [vocab.get(t, vocab["<unk>"]) for t in tokens]
    return torch.tensor(ids)

Batch Collation and DataLoader Setup

In [ ]:
def collate_fn(batch):
    texts = [encode(item["text"]) for item in batch]
    labels = torch.tensor([item["label"] for item in batch])

    # پدینگ برای یکسان کردن طول جمله‌ها
    texts = nn.utils.rnn.pad_sequence(texts, batch_first=True)

    return texts, labels


train_loader = DataLoader(train_data, batch_size=64, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False, collate_fn=collate_fn)

LSTM Model Definition (PyTorch)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()

        # embedding: تبدیل عددهای کلمات به بردارهای معنی‌دار
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        # FC: تبدیل hidden نهایی به خروجی 2 کلاسه (مثبت/منفی)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        # x شکلش (batch, seq_len)
        embedded = self.embedding(x)     # تبدیل به بردارهای معنی‌دار

        # LSTM دو خروجی دارد: output و (hidden, cell)
        _, (hidden, cell) = self.lstm(embedded)

        # hidden شکلش (1, batch, hidden_dim)
        hidden = hidden.squeeze(0)       # تبدیل به (batch, hidden_dim)

        # خروجی نهایی
        return self.fc(hidden)

Model Initialization for LSTM

In [ ]:
model = LSTMModel(len(vocab), embed_dim=256, hidden_dim=256).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Training Loop for LSTM

In [ ]:
for epoch in range(15):
    total_loss = 0

    for texts, labels in train_loader:
        # انتقال داده‌ها به GPU
        texts = texts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(texts)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Avg Loss: {avg_loss:.4f}")

Prediction Function for LSTM

In [ ]:
def predict(text):
    model.eval()
    with torch.no_grad():
        ids = encode(text).unsqueeze(0).to(device)
        output = model(ids)
        pred = torch.argmax(output).item()
        return "+" if pred == 1 else "-"

Test

In [ ]:
print(predict("This movie was absolutely amazing, I loved every minute of it."))
print(predict("This movie was terrible, boring and a complete waste of time."))